In [3]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize

# تحميل الـ NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# تحميل الداتا
df = pd.read_csv('medical_reports.csv') # تأكد إن اسم الملف مطابق للي عندك

print(df.head())
print(f"Shape: {df.shape}")

# التعديل الأول هنا: استخدام transcription بدل report
print(df['transcription'].str.len().describe())

def clean_medical_text(text):
    """تنظيف التقرير الطبي من الضوضاء"""
    # تحويل النص لـ string لتجنب أي إيرور لو فيه خلايا فاضية (NaN)
    text = str(text)
    
    # إزالة التواريخ 
    text = re.sub(
        r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|'
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{1,2},?\s+\d{4}\b',
        '', text, flags=re.IGNORECASE
    )
    # إزالة أكواد المستشفيات والـ IDs
    text = re.sub(r'\bID[-:]?\s*\w+\b|\bPT[-:]?\s*\w+\b', '', text)
    # إزالة الرموز الخاصة مع الحفاظ على نقاط الجمل
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\;\:\!\?]', ' ', text)
    # إزالة المسافات الزيادة
    text = re.sub(r'\s+', ' ', text).strip()
    # تحويل لـ lowercase
    text = text.lower()
    return text

def remove_stopwords(text):
    """إزالة الـ Stop Words مع الحفاظ على المصطلحات الطبية"""
    stop_words = set(stopwords.words('english'))
    medical_keep = {'no', 'not', 'without', 'normal', 'abnormal'}
    stop_words -= medical_keep

    words = word_tokenize(text)
    filtered = [w for w in words if w not in stop_words]
    return ' '.join(filtered)

# حذف أي صفوف مافيهاش نصوص طبية عشان نتجنب إيرورز التخطي
df = df.dropna(subset=['transcription'])

# التعديل الثاني هنا: تطبيق التنظيف على عمود transcription
df['clean_report'] = df['transcription'].apply(clean_medical_text)

print("✅ Preprocessing Done!")
print(df[['transcription', 'clean_report']].head(2))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Kimo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Kimo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Kimo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


   Unnamed: 0                                        description  \
0           0   A 23-year-old white female presents with comp...   
1           1           Consult for laparoscopic gastric bypass.   
2           2           Consult for laparoscopic gastric bypass.   
3           3                             2-D M-Mode. Doppler.     
4           4                                 2-D Echocardiogram   

             medical_specialty                                sample_name  \
0         Allergy / Immunology                         Allergic Rhinitis    
1                   Bariatrics   Laparoscopic Gastric Bypass Consult - 2    
2                   Bariatrics   Laparoscopic Gastric Bypass Consult - 1    
3   Cardiovascular / Pulmonary                    2-D Echocardiogram - 1    
4   Cardiovascular / Pulmonary                    2-D Echocardiogram - 2    

                                       transcription  \
0  SUBJECTIVE:,  This 23-year-old white female pr...   
1  PAST MEDICAL 

In [4]:
# ————— إصلاح المشاكل الـ 3 —————

# 1. معالجة الـ Missing Values
print(f"Missing values قبل: {df['transcription'].isna().sum()}")
df = df.dropna(subset=['transcription'])  # احذف الصفوف الفاضية
# أو بديل: df['transcription'].fillna('No report available', inplace=True)
print(f"Missing values بعد: {df['transcription'].isna().sum()}")

# 2. تحسين دالة التنظيف — إزالة الـ Section Headers الطبية
def clean_medical_text(text):
    # إزالة الـ headers زي SUBJECTIVE:, HISTORY:, ASSESSMENT:
    text = re.sub(r'\b[A-Z][A-Z\s\/]+:\s*,?\s*', ' ', text)
    # إزالة التواريخ
    text = re.sub(
        r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|'
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{1,2},?\s+\d{4}\b',
        '', text, flags=re.IGNORECASE
    )
    # إزالة أكواد المرضى والمستشفيات
    text = re.sub(r'\bID[-:]?\s*\w+\b|\bPT[-:]?\s*\w+\b', '', text)
    # إزالة الرموز الخاصة مع الحفاظ على نقاط الجمل
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\;\:\!\?]', ' ', text)
    # إزالة المسافات الزيادة
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    return text

# 3. استراتيجية التقطيع للتقارير الطويلة (chunk-based)
def smart_truncate(text, max_words=900):
    """قطّع النص بذكاء عند أقرب جملة مش في نص الكلمة"""
    words = text.split()
    if len(words) <= max_words:
        return text
    truncated = ' '.join(words[:max_words])
    # اقطع عند آخر نقطة عشان الجملة ماتكونش ناقصة
    last_dot = truncated.rfind('.')
    if last_dot > max_words * 4:  # لو النقطة في مكان معقول
        return truncated[:last_dot + 1]
    return truncated

# طبّق كل التحسينات
df['clean_report'] = df['transcription'].apply(clean_medical_text)
df['clean_report'] = df['clean_report'].apply(smart_truncate)

# تحقق من النتيجة
print(f"\nشكل الداتا النهائية: {df.shape}")
print(f"\nمثال على التنظيف المحسّن:")
print("قبل:", df['transcription'].iloc[0][:200])
print("\nبعد:", df['clean_report'].iloc[0][:200])

# إحصائيات الطول بعد التنظيف
print(f"\nمتوسط طول الـ clean_report: {df['clean_report'].str.split().str.len().mean():.0f} كلمة")
print(f"أقصى طول: {df['clean_report'].str.split().str.len().max()} كلمة")

Missing values قبل: 0
Missing values بعد: 0

شكل الداتا النهائية: (4966, 7)

مثال على التنظيف المحسّن:
قبل: SUBJECTIVE:,  This 23-year-old white female presents with complaint of allergies.  She used to have allergies when she lived in Seattle but she thinks they are worse here.  In the past, she has tried 

بعد: this 23 year old white female presents with complaint of allergies. she used to have allergies when she lived in seattle but she thinks they are worse here. in the past, she has tried claritin, and zy

متوسط طول الـ clean_report: 438 كلمة
أقصى طول: 900 كلمة


In [6]:
!pip install sumy

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.0 MB 2.5 MB/s eta 0:00:04
   ----- ---------------------------------- 1.0/8.0 MB 2.1 MB/s eta 0:00:04
   ------- -------------------------------- 1.6/8.0 MB 2.3 MB/s eta 0:00:03
   --------- ------------------------------ 1.8/8.0 MB 2.3 MB/s eta 0:00:03
   ----------- ---------------------------- 2.4/8.0 MB 2.1 MB/s eta 0:00:03
   ------------- -------------------------- 2.6/8.0 MB 2.1 MB/s eta 0:00:03
   --------------- ------------------------ 3.1/8.0 MB 2.1 MB/s eta 0:00:03
   ------------------ --------------------- 3.7/8.0 MB 2.1 MB/s eta 0:00:03
   -------------------- ------------------- 4.2/8.

  DEPRECATION: Building 'breadability' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'breadability'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'docopt' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'docopt'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [7]:
# ————— الخطوة 2: Extractive Summarization —————
# pip install sumy scikit-learn

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from nltk.tokenize import sent_tokenize

# ——— TF-IDF ———
def tfidf_summarize(text, n_sentences=3):
    sentences = sent_tokenize(text)
    if len(sentences) <= n_sentences:
        return text

    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(sentences)
    scores = np.array(tfidf_matrix.sum(axis=1)).flatten()

    top_idx = sorted(np.argsort(scores)[-n_sentences:])
    return ' '.join([sentences[i] for i in top_idx])

# ——— TextRank ———
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer

def textrank_summarize(text, n_sentences=3):
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = TextRankSummarizer()
    return ' '.join([str(s) for s in summarizer(parser.document, n_sentences)])

# ——— طبّق على أول 100 صف (للتجربة السريعة) ———
sample_df = df.head(100).copy()

sample_df['tfidf_summary']    = sample_df['clean_report'].apply(tfidf_summarize)
sample_df['textrank_summary'] = sample_df['clean_report'].apply(textrank_summarize)

# ——— شوف النتيجة ———
idx = 0
print("📄 النص الأصلي (أول 300 حرف):")
print(sample_df['clean_report'].iloc[idx][:300])
print("\n✂️  TF-IDF Summary:")
print(sample_df['tfidf_summary'].iloc[idx])
print("\n🌐 TextRank Summary:")
print(sample_df['textrank_summary'].iloc[idx])
print(f"\nضغط النص: {len(sample_df['clean_report'].iloc[idx].split())} كلمة → {len(sample_df['tfidf_summary'].iloc[idx].split())} كلمة")

📄 النص الأصلي (أول 300 حرف):
this 23 year old white female presents with complaint of allergies. she used to have allergies when she lived in seattle but she thinks they are worse here. in the past, she has tried claritin, and zyrtec. both worked for short time but then seemed to lose effectiveness. she has used allegra also. s

✂️  TF-IDF Summary:
this 23 year old white female presents with complaint of allergies. she does have asthma but doest not require daily medication for this and does not think it is flaring up., her only medication currently is ortho tri cyclen and the allegra., she has no known medicine allergies., vitals: weight was 130 pounds and blood pressure 124 78., her throat was mildly erythematous without exudate. tms were clear.,neck: supple without adenopathy.,lungs: clear., allergic rhinitis., 1. she will try zyrtec instead of allegra again.

🌐 TextRank Summary:
she used to have allergies when she lived in seattle but she thinks they are worse here. she does have a

In [8]:
# تنظيف الفاصلة الزيادة من الملخص
sample_df['tfidf_summary']    = sample_df['tfidf_summary'].str.replace(r',\.', '.', regex=True).str.replace(r'\.,', '.', regex=True)
sample_df['textrank_summary'] = sample_df['textrank_summary'].str.replace(r',\.', '.', regex=True).str.replace(r'\.,', '.', regex=True)

In [9]:
!pip install transformers torch sentencepiece

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   ----------------------------- ---------- 0.8/1.1 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 1.8 MB/s eta 0:00:00


In [10]:
# ————— الخطوة 3: Abstractive Summarization بـ BART —————
from transformers import pipeline

print("⏳ بيحمّل الموديل... (هياخد دقيقة أو 2 أول مرة)")
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=-1  # CPU — لو عندك GPU غيّرها لـ 0
)
print("✅ الموديل جاهز!")

⏳ بيحمّل الموديل... (هياخد دقيقة أو 2 أول مرة)


config.json: 0.00B [00:00, ?B/s]

C:\Users\Kimo\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kimo\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. Fo

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


✅ الموديل جاهز!


In [11]:
def abstractive_summarize(text, max_len=150, min_len=40):
    words = text.split()
    if len(words) > 900:
        text = ' '.join(words[:900])
    
    result = summarizer(
        text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False,
        num_beams=4,
        early_stopping=True
    )
    return result[0]['summary_text']

# جرّب على أول 5 تقارير الأول عشان نشوف النتيجة
sample_df['bart_summary'] = sample_df['clean_report'].head(5).apply(abstractive_summarize)

# شوف المقارنة
for i in range(3):
    print(f"\n{'='*60}")
    print(f"📄 التقرير الأصلي (أول 200 حرف):")
    print(sample_df['clean_report'].iloc[i][:200])
    print(f"\n✂️  TF-IDF:   {sample_df['tfidf_summary'].iloc[i][:200]}")
    print(f"\n🌐 TextRank: {sample_df['textrank_summary'].iloc[i][:200]}")
    print(f"\n🤖 BART:     {sample_df['bart_summary'].iloc[i]}")

Your max_length is set to 150, but your input_length is only 139. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=69)



📄 التقرير الأصلي (أول 200 حرف):
this 23 year old white female presents with complaint of allergies. she used to have allergies when she lived in seattle but she thinks they are worse here. in the past, she has tried claritin, and zy

✂️  TF-IDF:   this 23 year old white female presents with complaint of allergies. she does have asthma but doest not require daily medication for this and does not think it is flaring up. her only medication curren

🌐 TextRank: she used to have allergies when she lived in seattle but she thinks they are worse here. she does have asthma but doest not require daily medication for this and does not think it is flaring up. her o

🤖 BART:     23 year old white female presents with complaint of allergies. She used to have allergies when she lived in seattle but she thinks they are worse here. In the past, she has tried claritin, and zyrtec. Both worked for short time but then seemed to lose effectiveness.

📄 التقرير الأصلي (أول 200 حرف):
he has difficulty climb

In [12]:
!pip install rouge-score evaluate

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/529.0 kB ? eta -:--:--
   ---------------------------------------- 529.0/529.0 kB 2.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/27.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/27.3 MB ? eta -:--:--
   - -------------------------------------- 0.8/27.3 MB 2.5 MB/s eta 0:00:11
   - -------------------------------------- 1.3/27.3 MB 2.6 MB/s eta 0:00:10
   -- ------------------------------------- 1.8/27.3 MB 2.4 MB/s eta 0:00:11
   --- ------------------------------------ 2.4/27.3 MB 2.6 MB/s eta 0:00:10
   ---- ----------------------------------- 3.1/27.3 MB 2.7 MB/s eta 0:00:10
   ----- ---------------------------------- 3.7/27.3 MB 2.8 MB/s eta 0:00:09
   ------ --------------------------------- 4.5/27.3 MB 2.9 MB/s eta 0:0

  DEPRECATION: Building 'rouge-score' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'rouge-score'. Discussion can be found at https://github.com/pypa/pip/issues/6334
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires packaging<25,>=20, but you have packaging 26.1 which is incompatible.


In [13]:
# ————— الخطوة 4: ROUGE Evaluation —————
from rouge_score import rouge_scorer
import pandas as pd

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

def get_rouge_scores(prediction, reference):
    scores = scorer.score(reference, prediction)
    return {
        'ROUGE-1': round(scores['rouge1'].fmeasure * 100, 2),
        'ROUGE-2': round(scores['rouge2'].fmeasure * 100, 2),
        'ROUGE-L': round(scores['rougeL'].fmeasure * 100, 2),
    }

# هنستخدم TF-IDF كـ reference عشان نقارن TextRank و BART بيه
results = []
for i in range(5):
    ref  = sample_df['tfidf_summary'].iloc[i]
    tr   = get_rouge_scores(sample_df['textrank_summary'].iloc[i], ref)
    bart = get_rouge_scores(sample_df['bart_summary'].iloc[i], ref)
    results.append({
        'Report': i+1,
        'TextRank R1': tr['ROUGE-1'],
        'TextRank R2': tr['ROUGE-2'],
        'TextRank RL': tr['ROUGE-L'],
        'BART R1':     bart['ROUGE-1'],
        'BART R2':     bart['ROUGE-2'],
        'BART RL':     bart['ROUGE-L'],
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print(f"\n📊 متوسط الـ ROUGE Scores:")
print(f"TextRank → R1: {results_df['TextRank R1'].mean():.1f}%  R2: {results_df['TextRank R2'].mean():.1f}%  RL: {results_df['TextRank RL'].mean():.1f}%")
print(f"BART     → R1: {results_df['BART R1'].mean():.1f}%  R2: {results_df['BART R2'].mean():.1f}%  RL: {results_df['BART RL'].mean():.1f}%")

 Report  TextRank R1  TextRank R2  TextRank RL  BART R1  BART R2  BART RL
      1        65.88        63.10        65.88    34.85    16.92    28.79
      2        67.80        64.10        64.41    12.06     1.02     9.05
      3        48.36        42.15        41.80    15.38     2.22    12.09
      4        68.35        62.34        63.29    69.39    60.42    63.27
      5       100.00       100.00       100.00    28.57     3.88    19.05

📊 متوسط الـ ROUGE Scores:
TextRank → R1: 70.1%  R2: 66.3%  RL: 67.1%
BART     → R1: 32.0%  R2: 16.9%  RL: 26.5%


In [14]:
# ————— المقارنة الصح —————
# description = ملخص بشري حقيقي موجود في الداتا أصلًا!

results = []
for i in range(5):
    ref  = df['description'].iloc[i]  # الملخص البشري ← الـ Ground Truth
    
    tfidf_s   = get_rouge_scores(sample_df['tfidf_summary'].iloc[i],    ref)
    textrank_s = get_rouge_scores(sample_df['textrank_summary'].iloc[i], ref)
    bart_s    = get_rouge_scores(sample_df['bart_summary'].iloc[i],      ref)
    
    results.append({
        'Report':       i+1,
        'TF-IDF R1':    tfidf_s['ROUGE-1'],
        'TextRank R1':  textrank_s['ROUGE-1'],
        'BART R1':      bart_s['ROUGE-1'],
        'TF-IDF RL':    tfidf_s['ROUGE-L'],
        'TextRank RL':  textrank_s['ROUGE-L'],
        'BART RL':      bart_s['ROUGE-L'],
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print(f"\n📊 متوسط ROUGE-1 (vs Human Reference):")
print(f"  TF-IDF   → {results_df['TF-IDF R1'].mean():.1f}%")
print(f"  TextRank → {results_df['TextRank R1'].mean():.1f}%")
print(f"  BART     → {results_df['BART R1'].mean():.1f}%")

print(f"\n📊 متوسط ROUGE-L (vs Human Reference):")
print(f"  TF-IDF   → {results_df['TF-IDF RL'].mean():.1f}%")
print(f"  TextRank → {results_df['TextRank RL'].mean():.1f}%")
print(f"  BART     → {results_df['BART RL'].mean():.1f}%")

 Report  TF-IDF R1  TextRank R1  BART R1  TF-IDF RL  TextRank RL  BART RL
      1      20.83         2.08    34.48      20.83         2.08    34.48
      2       1.35         2.04     0.00       1.35         2.04     0.00
      3       5.93         1.68     3.51       5.93         1.68     3.51
      4       4.65        13.04     3.08       4.65        13.04     3.08
      5       0.00         0.00     0.00       0.00         0.00     0.00

📊 متوسط ROUGE-1 (vs Human Reference):
  TF-IDF   → 6.6%
  TextRank → 3.8%
  BART     → 8.2%

📊 متوسط ROUGE-L (vs Human Reference):
  TF-IDF   → 6.6%
  TextRank → 3.8%
  BART     → 8.2%


In [15]:
!pip install fastapi uvicorn nest-asyncio pyngrok


   -------------------- ------------------- 2/4 [starlette]
   -------------------- ------------------- 2/4 [starlette]
   ------------------------------ --------- 3/4 [fastapi]
   ------------------------------ --------- 3/4 [fastapi]
   ---------------------------------------- 4/4 [fastapi]



In [17]:
# ————— الخطوة 5: FastAPI محلياً بدون ngrok —————
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import sent_tokenize
import numpy as np
import re, threading

nest_asyncio.apply()

app = FastAPI(title="🏥 Medical Report Summarizer API")

class ReportRequest(BaseModel):
    report_text: str
    max_length: int = 150
    min_length: int = 40

class SummaryResponse(BaseModel):
    original_words: int
    summary_words: int
    compression_ratio: str
    bart_summary: str
    tfidf_summary: str

def clean_text(text: str) -> str:
    text = re.sub(r'\b[A-Z][A-Z\s\/]+:\s*,?\s*', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

def tfidf_summarize(text, n=3):
    sentences = sent_tokenize(text)
    if len(sentences) <= n:
        return text
    vec = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
    mat = vec.fit_transform(sentences)
    scores = np.array(mat.sum(axis=1)).flatten()
    top = sorted(np.argsort(scores)[-n:])
    return ' '.join([sentences[i] for i in top])

@app.get("/")
async def root():
    return {"status": "✅ API is running!", "model": "facebook/bart-large-cnn"}

@app.post("/summarize", response_model=SummaryResponse)
async def summarize_report(req: ReportRequest):
    if len(req.report_text.strip()) < 50:
        raise HTTPException(400, "النص قصير جداً — ادخل على الأقل 50 حرف")

    clean = clean_text(req.report_text)
    words = clean.split()
    if len(words) > 900:
        clean = ' '.join(words[:900])

    # BART
    bart_result = summarizer(
        clean,
        max_length=req.max_length,
        min_length=req.min_length,
        do_sample=False,
        num_beams=4,
        early_stopping=True
    )
    bart_sum = bart_result[0]['summary_text']

    # TF-IDF
    tfidf_sum = tfidf_summarize(clean)

    original_len = len(req.report_text.split())
    summary_len  = len(bart_sum.split())

    return SummaryResponse(
        original_words=original_len,
        summary_words=summary_len,
        compression_ratio=f"{round((1 - summary_len/original_len)*100)}%",
        bart_summary=bart_sum,
        tfidf_summary=tfidf_sum
    )

# ————— تشغيل السيرفر —————
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

thread = threading.Thread(target=run, daemon=True)
thread.start()

print("✅ API شغال على: http://localhost:8000/docs")
print("📬 جرّب الـ Endpoint: http://localhost:8000/summarize")

✅ API شغال على: http://localhost:8000/docs
📬 جرّب الـ Endpoint: http://localhost:8000/summarize


Your max_length is set to 150, but your input_length is only 121. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=60)
Your max_length is set to 150, but your input_length is only 121. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=60)


In [18]:
import requests, json

response = requests.post(
    "http://localhost:8000/summarize",
    json={
        "report_text": "SUBJECTIVE: This 23-year-old white female presents with complaint of allergies. She used to have allergies when she lived in Seattle but she thinks they are worse here. In the past, she has tried Claritin and Zyrtec. Both worked for short time but then seemed to lose effectiveness. She has used Allegra also. She does have asthma but does not require daily medication for this. Her only medication currently is Ortho Tri-Cyclen and Allegra. She has no known medicine allergies. ASSESSMENT: Allergic rhinitis. PLAN: She will try Zyrtec instead of Allegra again.",
        "max_length": 150,
        "min_length": 40
    }
)

print("✅ Status:", response.status_code)
print("\n📊 Response:")
print(json.dumps(response.json(), indent=2))

✅ Status: 200

📊 Response:
{
  "original_words": 91,
  "summary_words": 41,
  "compression_ratio": "55%",
  "bart_summary": "This 23-year-old white female presents with complaint of allergies. She used to have allergies when she lived in seattle but she thinks they are worse here. Her only medication currently is ortho tri-cyclen and allegra. She has no known medicine allergies.",
  "tfidf_summary": "this 23-year-old white female presents with complaint of allergies. she used to have allergies when she lived in seattle but she thinks they are worse here. her only medication currently is ortho tri-cyclen and allegra."
}


In [1]:
!pip install gradio

   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
    --------------------------------------- 0.3/19.7 MB ? eta -:--:--
   - -------------------------------------- 0.8/19.7 MB 1.7 MB/s eta 0:00:12
   -- ------------------------------------- 1.3/19.7 MB 2.0 MB/s eta 0:00:10
   --- ------------------------------------ 1.6/19.7 MB 2.0 MB/s eta 0:00:09
   ---- ----------------------------------- 2.4/19.7 MB 2.2 MB/s eta 0:00:08
   ----- ---------------------------------- 2.6/19.7 MB 2.2 MB/s eta 0:00:08
   ------ --------------------------------- 3.4/19.7 MB 2.3 MB/s eta 0:00:08
   ------- -------------------------------- 3.9/19.7 MB 2.4 MB/s eta 0:00:07
   --------- ------------------------------ 4.5/19.7 MB 2.4 MB/s eta 0:00:07
   ---------- ----------------------------- 5.2/19.7 MB 2.5 MB/s eta 0:00:06
   ----------- ---------------------------- 5.8/19.7 MB 2.5 MB/s eta 0:00:06
   ------------- ---

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires packaging<25,>=20, but you have packaging 26.1 which is incompatible.


In [2]:
# ————— MedBrief GUI بـ Gradio —————
import gradio as gr
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import sent_tokenize
import numpy as np
import re

# ——— دوال التنظيف والتلخيص ———
def clean_text(text):
    text = re.sub(r'\b[A-Z][A-Z\s\/]+:\s*,?\s*', ' ', text)
    text = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

def tfidf_summarize(text, n=3):
    sentences = sent_tokenize(text)
    if len(sentences) <= n:
        return text
    vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    mat = vec.fit_transform(sentences)
    scores = np.array(mat.sum(axis=1)).flatten()
    top = sorted(np.argsort(scores)[-n:])
    return ' '.join([sentences[i] for i in top])

def textrank_summarize(text, n=3):
    from sumy.parsers.plaintext import PlaintextParser
    from sumy.nlp.tokenizers import Tokenizer
    from sumy.summarizers.text_rank import TextRankSummarizer
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = TextRankSummarizer()
    return ' '.join([str(s) for s in summarizer(parser.document, n)])

def summarize(report_text, method, max_length, min_length):
    if len(report_text.strip()) < 50:
        return "❌ النص قصير جداً — ادخل على الأقل 50 حرف", "", ""

    clean = clean_text(report_text)
    words = clean.split()
    if len(words) > 900:
        clean = ' '.join(words[:900])

    original_words = len(report_text.split())

    if method == "BART (Abstractive)":
        result = summarizer(
            clean,
            max_length=int(max_length),
            min_length=int(min_length),
            do_sample=False,
            num_beams=4,
            early_stopping=True
        )
        summary = result[0]['summary_text']

    elif method == "TF-IDF (Extractive)":
        summary = tfidf_summarize(clean)

    else:
        summary = textrank_summarize(clean)

    summary_words  = len(summary.split())
    compression    = round((1 - summary_words / original_words) * 100)
    stats = f"📄 الأصلي: {original_words} كلمة   →   ✂️ الملخص: {summary_words} كلمة   |   🗜️ ضغط: {compression}%"

    return summary, stats

# ——— واجهة Gradio ———
with gr.Blocks(title="MedBrief 🏥", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🏥 MedBrief
    ### AI-Powered Medical Report Summarizer
    ادخل التقرير الطبي واختار طريقة التلخيص
    """)

    with gr.Row():
        with gr.Column(scale=2):
            input_text = gr.Textbox(
                label="📋 التقرير الطبي",
                placeholder="الصق التقرير الطبي هنا...",
                lines=12
            )
            with gr.Row():
                method = gr.Radio(
                    choices=["BART (Abstractive)", "TF-IDF (Extractive)", "TextRank (Extractive)"],
                    value="BART (Abstractive)",
                    label="🤖 طريقة التلخيص"
                )
            with gr.Row():
                max_len = gr.Slider(50, 300, value=150, step=10, label="الحد الأقصى للملخص")
                min_len = gr.Slider(20, 100, value=40,  step=10, label="الحد الأدنى للملخص")

            btn = gr.Button("🚀 لخّص التقرير", variant="primary", size="lg")

        with gr.Column(scale=2):
            output_summary = gr.Textbox(
                label="📝 الملخص",
                lines=10,
                show_copy_button=True
            )
            output_stats = gr.Textbox(label="📊 الإحصائيات", interactive=False)

    # ——— أمثلة جاهزة ———
    gr.Examples(
        examples=[
            ["SUBJECTIVE: This 23-year-old white female presents with complaint of allergies. She used to have allergies when she lived in Seattle but she thinks they are worse here. In the past, she has tried Claritin and Zyrtec. Both worked for short time but then seemed to lose effectiveness. She has used Allegra also. She does have asthma but does not require daily medication. ASSESSMENT: Allergic rhinitis. PLAN: She will try Zyrtec instead of Allegra again.", "BART (Abstractive)", 150, 40],
        ],
        inputs=[input_text, method, max_len, min_len]
    )

    btn.click(
        fn=summarize,
        inputs=[input_text, method, max_len, min_len],
        outputs=[output_summary, output_stats]
    )

demo.launch(share=False, server_port=7860)
print("🌐 MedBrief GUI شغال على: http://localhost:7860")

C:\Users\Kimo\AppData\Local\Temp\ipykernel_10020\2212752694.py:69: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="MedBrief 🏥", theme=gr.themes.Soft()) as demo:


TypeError: Textbox.__init__() got an unexpected keyword argument 'show_copy_button'

In [3]:
with gr.Blocks(title="MedBrief 🏥") as demo:

    gr.Markdown("""
    # 🏥 MedBrief
    ### AI-Powered Medical Report Summarizer
    ادخل التقرير الطبي واختار طريقة التلخيص
    """)

    with gr.Row():
        with gr.Column(scale=2):
            input_text = gr.Textbox(
                label="📋 التقرير الطبي",
                placeholder="الصق التقرير الطبي هنا...",
                lines=12
            )
            with gr.Row():
                method = gr.Radio(
                    choices=["BART (Abstractive)", "TF-IDF (Extractive)", "TextRank (Extractive)"],
                    value="BART (Abstractive)",
                    label="🤖 طريقة التلخيص"
                )
            with gr.Row():
                max_len = gr.Slider(50, 300, value=150, step=10, label="الحد الأقصى للملخص")
                min_len = gr.Slider(20, 100, value=40,  step=10, label="الحد الأدنى للملخص")

            btn = gr.Button("🚀 لخّص التقرير", variant="primary")

        with gr.Column(scale=2):
            output_summary = gr.Textbox(
                label="📝 الملخص",
                lines=10
            )
            output_stats = gr.Textbox(label="📊 الإحصائيات", interactive=False)

    gr.Examples(
        examples=[
            ["SUBJECTIVE: This 23-year-old white female presents with complaint of allergies. She used to have allergies when she lived in Seattle but she thinks they are worse here. In the past, she has tried Claritin and Zyrtec. Both worked for short time but then seemed to lose effectiveness. She has used Allegra also. She does have asthma but does not require daily medication. ASSESSMENT: Allergic rhinitis. PLAN: She will try Zyrtec instead of Allegra again.", "BART (Abstractive)", 150, 40],
        ],
        inputs=[input_text, method, max_len, min_len]
    )

    btn.click(
        fn=summarize,
        inputs=[input_text, method, max_len, min_len],
        outputs=[output_summary, output_stats]
    )

demo.launch(
    share=False,
    server_port=7860,
    theme=gr.themes.Soft()
)
print("🌐 MedBrief GUI شغال على: http://localhost:7860")

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


🌐 MedBrief GUI شغال على: http://localhost:7860


Traceback (most recent call last):
  File "C:\Users\Kimo\anaconda3\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\Kimo\anaconda3\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "C:\Users\Kimo\anaconda3\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "C:\Users\Kimo\anaconda3\Lib\site-packages\gradio\blocks.py", line 1636, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^

In [4]:
# ————— MedBrief GUI — الكود الكامل في cell واحدة —————
import gradio as gr
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import sent_tokenize
import numpy as np
import re

# تحميل الموديل
print("⏳ بيحمّل BART...")
bart_model = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=-1
)
print("✅ BART جاهز!")

# ——— دوال التنظيف ———
def clean_text(text):
    text = re.sub(r'\b[A-Z][A-Z\s\/]+:\s*,?\s*', ' ', text)
    text = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

def tfidf_summarize(text, n=3):
    sentences = sent_tokenize(text)
    if len(sentences) <= n:
        return text
    vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    mat = vec.fit_transform(sentences)
    scores = np.array(mat.sum(axis=1)).flatten()
    top = sorted(np.argsort(scores)[-n:])
    return ' '.join([sentences[i] for i in top])

def textrank_summarize(text, n=3):
    from sumy.parsers.plaintext import PlaintextParser
    from sumy.nlp.tokenizers import Tokenizer
    from sumy.summarizers.text_rank import TextRankSummarizer
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = TextRankSummarizer()
    return ' '.join([str(s) for s in summarizer(parser.document, n)])

# ——— دالة التلخيص الرئيسية ———
def summarize(report_text, method, max_length, min_length):
    if len(report_text.strip()) < 50:
        return "❌ النص قصير جداً — ادخل على الأقل 50 حرف", ""

    clean = clean_text(report_text)
    words = clean.split()
    if len(words) > 900:
        clean = ' '.join(words[:900])

    original_words = len(report_text.split())

    if method == "BART (Abstractive)":
        result = bart_model(
            clean,
            max_length=int(max_length),
            min_length=int(min_length),
            do_sample=False,
            num_beams=4,
            early_stopping=True
        )
        summary = result[0]['summary_text']

    elif method == "TF-IDF (Extractive)":
        summary = tfidf_summarize(clean)

    else:
        summary = textrank_summarize(clean)

    summary_words = len(summary.split())
    compression   = round((1 - summary_words / original_words) * 100)
    stats = f"📄 الأصلي: {original_words} كلمة   →   ✂️ الملخص: {summary_words} كلمة   |   🗜️ ضغط: {compression}%"

    return summary, stats

# ——— واجهة Gradio ———
with gr.Blocks(title="MedBrief 🏥") as demo:

    gr.Markdown("""
    # 🏥 MedBrief
    ### AI-Powered Medical Report Summarizer
    ادخل التقرير الطبي واختار طريقة التلخيص
    """)

    with gr.Row():
        with gr.Column(scale=2):
            input_text = gr.Textbox(
                label="📋 التقرير الطبي",
                placeholder="الصق التقرير الطبي هنا...",
                lines=12
            )
            method = gr.Radio(
                choices=["BART (Abstractive)", "TF-IDF (Extractive)", "TextRank (Extractive)"],
                value="BART (Abstractive)",
                label="🤖 طريقة التلخيص"
            )
            with gr.Row():
                max_len = gr.Slider(50, 300, value=150, step=10, label="الحد الأقصى للملخص")
                min_len = gr.Slider(20, 100, value=40,  step=10, label="الحد الأدنى للملخص")

            btn = gr.Button("🚀 لخّص التقرير", variant="primary")

        with gr.Column(scale=2):
            output_summary = gr.Textbox(label="📝 الملخص", lines=10)
            output_stats   = gr.Textbox(label="📊 الإحصائيات", interactive=False)

    gr.Examples(
        examples=[
            ["SUBJECTIVE: This 23-year-old white female presents with complaint of allergies. She used to have allergies when she lived in Seattle but she thinks they are worse here. In the past, she has tried Claritin and Zyrtec. Both worked for short time but then seemed to lose effectiveness. ASSESSMENT: Allergic rhinitis. PLAN: She will try Zyrtec instead of Allegra again.", "BART (Abstractive)", 150, 40],
        ],
        inputs=[input_text, method, max_len, min_len]
    )

    btn.click(
        fn=summarize,
        inputs=[input_text, method, max_len, min_len],
        outputs=[output_summary, output_stats]
    )

demo.launch(share=False, server_port=7860, theme=gr.themes.Soft())

⏳ بيحمّل BART...


Device set to use cpu


✅ BART جاهز!


OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.

In [5]:
demo.launch(share=False, server_port=7861, theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
